# Python Pandas Exercises: 10 Coding Problems with Solutions

A practice notebook on pandas DataFrames using a real automobile dataset — loading, cleaning, filtering, `groupby`, sorting, concatenation, and merging — each with a concept note, a hint, a solution, and an explanation.

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/python-pandas-exercise/), using the same [Automobile Dataset](https://pynative.com/wp-content/uploads/2019/01/Automobile_data.csv) (embedded directly in Exercise 1's cell so the notebook runs standalone). Exercises 6 and 7 had a real syntax bug on the source page — confirmed by reader comments — which is fixed here along with a related correctness issue in Exercise 6 that another reader flagged.*

---

## Concepts you'll need

This set uses a real automobile dataset to practice **pandas**, the core data-analysis library built on top of NumPy.

- **Loading data** — `pd.read_csv(path)` reads a CSV into a `DataFrame`; the `na_values=` argument lets you specify per-column placeholder strings (like `"?"`) that should be treated as missing (`NaN`) during loading, rather than cleaning them up afterward.
- **Peeking at data** — `.head(n)` / `.tail(n)` show the first/last n rows without printing the whole (possibly huge) DataFrame.
- **Selecting rows by condition (boolean indexing)** — `df[df['price'] == df['price'].max()]` first computes a boolean Series (True/False per row), then uses it to filter — the same pattern as NumPy boolean masking, extended to DataFrames. Select specific columns at the same time with `df[['col1', 'col2']][condition]`.
- **`.groupby(column)`** — splits the DataFrame into groups sharing the same value in that column; `.get_group(name)` retrieves one specific group as its own DataFrame. Aggregate every group at once with `.mean()`, `.max()`, `.sum()`, etc.
- **A `groupby` + `max` pitfall** — `grouped[['col_a','col_b']].max()` takes the max of *each column independently per group*, which can silently mix values from different original rows. When you need the *entire row* containing a group's maximum, use `.loc[grouped['price'].idxmax()]` instead, which finds the actual row index of the max and selects that whole row.
- **`.value_counts()`** — counts how many times each unique value appears in a column, returned already sorted from most to least common.
- **`.sort_values(by=[...], ascending=...)`** — sorts by one or more columns; a list of columns sorts by the first, using later columns only to break ties.
- **Combining DataFrames** — `pd.concat([df1, df2], keys=[...])` stacks DataFrames on top of each other (optionally tagging which original DataFrame each row came from); `pd.merge(df1, df2, on='column')` joins two DataFrames side-by-side by matching values in a shared key column, similar to a SQL join.

Each exercise below gives a problem, a hint, a solution, and an explanation. The dataset is embedded directly in the first exercise's cell so the whole notebook runs standalone.

**Note:** Exercises 6 and 7 had a syntax bug on the source page — `grouped['col1','col2'].max()` (selecting multiple columns with a plain tuple-like comma) stopped working in modern pandas and raises a `KeyError`. Both are fixed here using the correct double-bracket syntax `grouped[['col1','col2']]`, and Exercise 6 additionally uses `.idxmax()` to correctly return each company's actual highest-priced row, addressing a bug a reader also flagged in the comments.

## Exercise 1. Print the First and Last Five Rows

**Concept:** pd.read_csv(), .head(), .tail()

**Problem:** Load the automobile dataset and print its first 5 and last 5 rows.

**Given:**
```
Automobile_data.csv (43 columns... actually 10 columns, ~55 rows)
```

**Expected Output:**
```
First 5 rows and last 5 rows of the dataset, as formatted tables
```

**Hint:** .head(n) and .tail(n) both default to n=5 if no argument is given.

In [ ]:
import pandas as pd
import io

# The dataset embedded directly so this notebook is fully self-contained
csv_data = """index,company,body-style,wheel-base,length,engine-type,num-of-cylinders,horsepower,average-mileage,price
0,alfa-romero,convertible,88.6,168.8,dohc,four,111,21,13495
1,alfa-romero,convertible,88.6,168.8,dohc,four,111,21,16500
2,alfa-romero,hatchback,94.5,171.2,ohcv,six,154,19,16500
3,audi,sedan,99.8,176.6,ohc,four,102,24,13950
4,audi,sedan,99.4,176.6,ohc,five,115,18,17450
5,audi,sedan,99.8,177.3,ohc,five,110,19,15250
6,audi,wagon,105.8,192.7,ohc,five,110,19,18920
9,bmw,sedan,101.2,176.8,ohc,four,101,23,16430
10,bmw,sedan,101.2,176.8,ohc,four,101,23,16925
11,bmw,sedan,101.2,176.8,ohc,six,121,21,20970
13,bmw,sedan,103.5,189,ohc,six,182,16,30760
14,bmw,sedan,103.5,193.8,ohc,six,182,16,41315
15,bmw,sedan,110,197,ohc,six,182,15,36880
16,chevrolet,hatchback,88.4,141.1,l,three,48,47,5151
17,chevrolet,hatchback,94.5,155.9,ohc,four,70,38,6295
18,chevrolet,sedan,94.5,158.8,ohc,four,70,38,6575
19,dodge,hatchback,93.7,157.3,ohc,four,68,31,6377
20,dodge,hatchback,93.7,157.3,ohc,four,68,31,6229
27,honda,wagon,96.5,157.1,ohc,four,76,30,7295
28,honda,sedan,96.5,175.4,ohc,four,101,24,12945
29,honda,sedan,96.5,169.1,ohc,four,100,25,10345
30,isuzu,sedan,94.3,170.7,ohc,four,78,24,6785
31,isuzu,sedan,94.5,155.9,ohc,four,70,38,
32,isuzu,sedan,94.5,155.9,ohc,four,70,38,
33,jaguar,sedan,113,199.6,dohc,six,176,15,32250
34,jaguar,sedan,113,199.6,dohc,six,176,15,35550
35,jaguar,sedan,102,191.7,ohcv,twelve,262,13,36000
36,mazda,hatchback,93.1,159.1,ohc,four,68,30,5195
37,mazda,hatchback,93.1,159.1,ohc,four,68,31,6095
38,mazda,hatchback,93.1,159.1,ohc,four,68,31,6795
39,mazda,hatchback,95.3,169,rotor,two,101,17,11845
43,mazda,sedan,104.9,175,ohc,four,72,31,18344
44,mercedes-benz,sedan,110,190.9,ohc,five,123,22,25552
45,mercedes-benz,wagon,110,190.9,ohc,five,123,22,28248
46,mercedes-benz,sedan,120.9,208.1,ohcv,eight,184,14,40960
47,mercedes-benz,hardtop,112,199.2,ohcv,eight,184,14,45400
49,mitsubishi,hatchback,93.7,157.3,ohc,four,68,37,5389
50,mitsubishi,hatchback,93.7,157.3,ohc,four,68,31,6189
51,mitsubishi,sedan,96.3,172.4,ohc,four,88,25,6989
52,mitsubishi,sedan,96.3,172.4,ohc,four,88,25,8189
53,nissan,sedan,94.5,165.3,ohc,four,55,45,7099
54,nissan,sedan,94.5,165.3,ohc,four,69,31,6649
55,nissan,sedan,94.5,165.3,ohc,four,69,31,6849
56,nissan,wagon,94.5,170.2,ohc,four,69,31,7349
57,nissan,sedan,100.4,184.6,ohcv,six,152,19,13499
61,porsche,hardtop,89.5,168.9,ohcf,six,207,17,34028
62,porsche,convertible,89.5,168.9,ohcf,six,207,17,37028
63,porsche,hatchback,98.4,175.7,dohcv,eight,288,17,
66,toyota,hatchback,95.7,158.7,ohc,four,62,35,5348
67,toyota,hatchback,95.7,158.7,ohc,four,62,31,6338
68,toyota,hatchback,95.7,158.7,ohc,four,62,31,6488
69,toyota,wagon,95.7,169.7,ohc,four,62,31,6918
70,toyota,wagon,95.7,169.7,ohc,four,62,27,7898
71,toyota,wagon,95.7,169.7,ohc,four,62,27,8778
79,toyota,wagon,104.5,187.8,dohc,six,156,19,15750
80,volkswagen,sedan,97.3,171.7,ohc,four,52,37,7775
81,volkswagen,sedan,97.3,171.7,ohc,four,85,27,7975
82,volkswagen,sedan,97.3,171.7,ohc,four,52,37,7995
86,volkswagen,sedan,97.3,171.7,ohc,four,100,26,9995
87,volvo,sedan,104.3,188.8,ohc,four,114,23,12940
88,volvo,wagon,104.3,188.8,ohc,four,114,23,13415
"""

with open("Automobile_data.csv", "w") as f:
    f.write(csv_data)

df = pd.read_csv("Automobile_data.csv")

print("First 5 rows:")
print(df.head(5))

print("\nLast 5 rows:")
print(df.tail(5))

**Explanation:** pd.read_csv() parses the CSV text into a DataFrame — pandas' primary 2D, labeled data structure, with rows and named columns. .head(5) shows the first 5 rows, and .tail(5) shows the last 5, both useful for a quick sanity check of a dataset's shape and content without printing potentially thousands of rows at once.

## Exercise 2. Clean the Dataset (Handle Missing Values)

**Concept:** na_values= in read_csv(), for treating placeholder strings as NaN

**Problem:** Load the dataset, treating '?' as a missing value in the price column, and check which rows are affected.

**Given:**
```
the price column contains '?' for a few rows with genuinely unknown prices
```

**Expected Output:**
```
Rows with missing prices flagged, and a summary of how many missing values exist per column
```

**Hint:** na_values can be a dict, letting you specify a DIFFERENT set of placeholder strings for each individual column.

In [ ]:
import pandas as pd

df = pd.read_csv("Automobile_data.csv", na_values={
    'price': ["?", "n.a"],
    'horsepower': ["?", "n.a"],
    'average-mileage': ["?", "n.a"],
})

print("Missing values per column:")
print(df.isnull().sum())

print("\nRows with a missing price:")
print(df[df['price'].isnull()][['company', 'body-style', 'price']])

**Explanation:** Passing a dict to na_values lets each named column have its own list of strings that should be recognized as missing data DURING loading — here '?' in the price, horsepower, or average-mileage columns becomes a genuine NaN rather than an unparseable string. .isnull().sum() counts missing values per column in one call, and .isnull() used as a boolean mask (df[df['price'].isnull()]) isolates exactly the rows affected, letting you inspect or handle them (drop, fill, or otherwise) explicitly.

## Exercise 3. Find the Most Expensive Car

**Concept:** boolean filtering combined with a column-wide aggregate (.max())

**Problem:** Find and print the company name and price of the most expensive car in the dataset.

**Given:**
```
the full dataset
```

**Expected Output:**
```
company: mercedes-benz, price: 45400
```

**Hint:** df['price'] == df['price'].max() builds a boolean mask identifying the single row (or rows) matching the overall maximum price.

In [ ]:
import pandas as pd

df = pd.read_csv("Automobile_data.csv")

most_expensive = df[['company', 'price']][df.price == df['price'].max()]
print(most_expensive)

**Explanation:** df['price'].max() first computes the single highest price value across the WHOLE column. df.price == df['price'].max() then compares every row's price against that value, producing a boolean mask that's True only for the row(s) matching the maximum exactly. df[['company', 'price']][mask] applies two selections together: which COLUMNS to show ([['company', 'price']]) and which ROWS to include (the boolean mask).

## Exercise 4. Print All Toyota Cars' Details

**Concept:** .groupby() combined with .get_group()

**Problem:** Print all rows belonging to the 'toyota' company.

**Given:**
```
the full dataset
```

**Expected Output:**
```
All 7 Toyota rows from the dataset
```

**Hint:** groupby() itself doesn't filter anything by itself — get_group(name) retrieves one specific group as its own DataFrame.

In [ ]:
import pandas as pd

df = pd.read_csv("Automobile_data.csv")

car_Manufacturers = df.groupby('company')
toyotaDf = car_Manufacturers.get_group('toyota')
print(toyotaDf)

# A simpler, more direct alternative for a ONE-TIME filter like this:
print("\nEquivalent using boolean filtering directly:")
print(df[df['company'] == 'toyota'])

**Explanation:** df.groupby('company') splits the DataFrame into a collection of sub-groups, one per unique company value, without yet materializing any of them individually. .get_group('toyota') retrieves specifically the 'toyota' group as its own standalone DataFrame. For a single one-off filter like this, plain boolean indexing (df[df['company'] == 'toyota']) is simpler and more direct — groupby() earns its keep when you need to compute something across EVERY group at once, as the remaining exercises show.

## Exercise 5. Count Total Cars per Company

**Concept:** .value_counts()

**Problem:** Count how many car listings each company has in the dataset.

**Given:**
```
the full dataset
```

**Expected Output:**
```
A count of rows per company, sorted from most to least
```

**Hint:** value_counts() is a Series method — call it on a single column, not the whole DataFrame.

In [ ]:
import pandas as pd

df = pd.read_csv("Automobile_data.csv")

print(df['company'].value_counts())

**Explanation:** df['company'] selects just the company column as a pandas Series (a single labeled column of data). .value_counts() tallies how many times each unique value appears in that Series, and returns the result already sorted from most frequent to least frequent — no separate .sort_values() call needed.

## Exercise 6. Find Each Company's Highest-Priced Car

**Concept:** groupby + idxmax() to get the CORRECT full row, not per-column maxes

**Problem:** Find each company's single highest-priced car, showing the complete row for that specific car.

**Given:**
```
the full dataset
```

**Expected Output:**
```
One row per company: the exact car (with all its original column values) that has that company's highest price
```

**Hint:** grouped[['col1','col2']].max() computes each column's max INDEPENDENTLY, which can mix values from different rows — idxmax() avoids this by finding the actual row index of the max.

In [ ]:
import pandas as pd

df = pd.read_csv("Automobile_data.csv")

car_Manufacturers = df.groupby('company')

# Correct approach: idxmax() finds the actual row-index of each group's
# max price, then .loc[] retrieves that whole row intact.
highest_price_idx = car_Manufacturers['price'].idxmax()
priceDf = df.loc[highest_price_idx]
print(priceDf[['company', 'body-style', 'horsepower', 'price']])

**Explanation:** A naive approach — car_Manufacturers[['company','price']].max() — computes the maximum of EACH selected column independently within each group; if you also included a third column like 'horsepower', that column's max might come from an entirely DIFFERENT row than the one holding the max price, silently mixing data from unrelated cars into one misleading row. idxmax() instead returns the actual DataFrame INDEX of the row holding each group's maximum price. Passing those indices to df.loc[] retrieves each group's complete, correct original row — company, body-style, horsepower, and price all guaranteed to come from the SAME car.

## Exercise 7. Find the Average Mileage of Each Company

**Concept:** groupby + .mean() on a single column

**Problem:** Find the average mileage for cars from each company.

**Given:**
```
the full dataset
```

**Expected Output:**
```
Average mileage per company
```

**Hint:** Select the single target column from the grouped object BEFORE calling .mean(), for a clean Series result.

In [ ]:
import pandas as pd

df = pd.read_csv("Automobile_data.csv")

car_Manufacturers = df.groupby('company')
mileageDf = car_Manufacturers['average-mileage'].mean()
print(mileageDf)

**Explanation:** car_Manufacturers['average-mileage'] selects just that one column from the grouped object — the source page's original code instead tried grouped['company','average-mileage'], a tuple-like comma syntax for multi-column selection that raises a KeyError in modern pandas (which requires double brackets, [['col1','col2']], for multiple columns). .mean() then computes the average within each group independently, returning one mean value per company as a clean, readable Series.

## Exercise 8. Sort All Cars by Price

**Concept:** .sort_values() with multiple sort columns

**Problem:** Sort all cars by price (descending), using horsepower as a tiebreaker, and show the top 5.

**Given:**
```
the full dataset
```

**Expected Output:**
```
The 5 most expensive cars, with horsepower used to break any price ties
```

**Hint:** A list of columns in `by=` sorts primarily by the first column; later columns only matter to break ties in earlier ones.

In [ ]:
import pandas as pd

carsDf = pd.read_csv("Automobile_data.csv")
carsDf = carsDf.sort_values(by=['price', 'horsepower'], ascending=False)
print(carsDf[['company', 'price', 'horsepower']].head(5))

**Explanation:** by=['price', 'horsepower'] sorts PRIMARILY by price; horsepower only comes into play to break ties between rows that happen to share the exact same price. ascending=False reverses the default (lowest-first) order for BOTH sort columns, putting the highest prices (and, among ties, the highest horsepower) first. .head(5) then trims the sorted result down to just the top 5 rows.

## Exercise 9. Concatenate Two DataFrames

**Concept:** pd.concat() with keys= to tag each source DataFrame

**Problem:** Combine two separate DataFrames (German and Japanese car prices) into one, tagging which came from which.

**Given:**
```
GermanCars = {"Company": [...], "Price": [...]}, japaneseCars = {"Company": [...], "Price": [...]}
```

**Expected Output:**
```
A combined DataFrame with a two-level index: (Germany/Japan, original row number)
```

**Hint:** keys= adds an extra OUTER index level, labeled with whichever source each row came from.

In [ ]:
import pandas as pd

GermanCars = {'Company': ['Ford', 'Mercedes', 'BMW', 'Audi'], 'Price': [23845, 171995, 135925, 71400]}
carsDf1 = pd.DataFrame.from_dict(GermanCars)

japaneseCars = {'Company': ['Toyota', 'Honda', 'Nissan', 'Mitsubishi'], 'Price': [29995, 23600, 61500, 58900]}
carsDf2 = pd.DataFrame.from_dict(japaneseCars)

carsDf = pd.concat([carsDf1, carsDf2], keys=["Germany", "Japan"])
print(carsDf)

**Explanation:** pd.DataFrame.from_dict() converts a plain Python dictionary of column-name-to-value-list pairs directly into a DataFrame. pd.concat([carsDf1, carsDf2]) stacks the two DataFrames vertically, one after another. The keys=["Germany", "Japan"] argument adds an extra OUTER level to the resulting index, so every row is now labeled with both which country it came from AND its original row number within that source DataFrame — useful for tracing a combined row back to its origin.

## Exercise 10. Merge Two DataFrames

**Concept:** pd.merge() — a SQL-style join on a shared key column

**Problem:** Merge a price DataFrame and a horsepower DataFrame into one, matching rows by company name.

**Given:**
```
Car_Price and car_Horsepower, both keyed by Company
```

**Expected Output:**
```
One combined DataFrame with Company, Price, AND horsepower columns together
```

**Hint:** on='Company' tells merge() which column identifies matching rows between the two DataFrames.

In [ ]:
import pandas as pd

Car_Price = {'Company': ['Toyota', 'Honda', 'BMW', 'Audi'], 'Price': [23845, 17995, 135925, 71400]}
carPriceDf = pd.DataFrame.from_dict(Car_Price)

car_Horsepower = {'Company': ['Toyota', 'Honda', 'BMW', 'Audi'], 'horsepower': [141, 80, 182, 160]}
carsHorsepowerDf = pd.DataFrame.from_dict(car_Horsepower)

carsDf = pd.merge(carPriceDf, carsHorsepowerDf, on="Company")
print(carsDf)

**Explanation:** Unlike pd.concat() (which stacks DataFrames on top of each other), pd.merge() joins them SIDE BY SIDE, matching rows based on a shared key column — here, on="Company" tells pandas to line up rows from both DataFrames wherever their Company values match, similar to a SQL INNER JOIN. The result is a single DataFrame combining Price (from the first source) and horsepower (from the second) into one row per company, with Company appearing only once rather than duplicated from both sides.